# Generate Persian Conditionals with BIO Span Annotations

This notebook uses an OpenAI-compatible chat model (configured below as **GPT 6 Luna**) to generate
Persian sentences containing conditional constructions, and asks the model to identify the
**protasis** (the condition clause, label `IF`) and the **apodosis** (the consequent clause,
label `THEN`) as exact substrings of the generated text.

Two prompt variants are used, alternating example by example, and each generated example records
which one produced it (`prompt_variant`):

- **`marked`** — the sentence must use an explicit conditional marker word (`اگر`, `هرگاه`,
  `در صورتی که`, `چنانچه`, `به شرطی که`, ...).
- **`unmarked`** — the sentence must express a conditional relation *without* any such marker
  word (e.g. imperative + result juxtaposition, paratactic "do X, Y happens" constructions, or
  verb-mood-based conditionals).

Those substrings are located in the text (character offsets), the text is tokenized, and the
spans are converted into standard **BIO tags** (`B-IF`, `I-IF`, `B-THEN`, `I-THEN`, `O`) — one tag
per token, the classic sequence-labeling format used for CoNLL-style span annotation.

**Requirements**
- `pip install openai` (and optionally `python-dotenv`)
- An API key in the `OPENAI_API_KEY` environment variable (set it in your shell, or in a `.env`
  file next to this notebook — the notebook will try to load it with `python-dotenv` if available).

In [1]:
import itertools
import json
import os
import random
import re

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from openai import OpenAI
from tqdm.auto import tqdm

c:\Users\lischkaf\micromamba\envs\nlp-fall-school\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

`MODEL_NAME` and `BASE_URL` are the two knobs to point this at "GPT 6 Luna" — set `BASE_URL` if
it's served behind a custom/proxy endpoint rather than the default OpenAI API host.

In [2]:
CONFIG = {
    "model_name": os.environ.get("CONDITIONALS_MODEL_NAME", "gpt-6-luna"),
    "base_url": os.environ.get("OPENAI_BASE_URL"),  # None -> default OpenAI endpoint
    "num_examples": 1000,
    "variants": ["marked", "unmarked"],  # alternated example by example
    "temperature": 1.0,
    "max_retries": 3,
    "random_seed": 42,
    "output_path": "persian_conditionals_bio.jsonl",
}

random.seed(CONFIG["random_seed"])

if "OPENAI_API_KEY" not in os.environ:
    raise RuntimeError(
        "Set the OPENAI_API_KEY environment variable before running this notebook "
        "(e.g. in your shell, or in a .env file next to this notebook)."
    )

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=CONFIG["base_url"])

## Tokenizer and BIO conversion

A simple regex tokenizer: runs of word characters (including the Persian ZWNJ `\u200c` used in
compound words like `می‌گفت`) are one token; every other non-space character (punctuation) is its
own token. Each token carries its character offsets so spans can be matched by overlap.

In [3]:
TOKEN_PATTERN = re.compile(r"[\w\u200c]+|[^\s\w]", re.UNICODE)


def tokenize_with_offsets(text):
    """Return a list of (token, start, end) for `text`."""
    return [(m.group(), m.start(), m.end()) for m in TOKEN_PATTERN.finditer(text)]


def locate_spans(text, raw_spans):
    """Resolve {"label", "text"} spans (verbatim substrings) to char offsets, sorted and
    checked for overlap. Raises ValueError if a span can't be found or spans overlap."""
    located = []
    for rs in raw_spans:
        substr = rs["text"].strip()
        if not substr:
            continue
        start = text.find(substr)
        if start == -1:
            raise ValueError(f"span text not found verbatim in generated text: {substr!r}")
        located.append({"label": rs["label"], "start": start, "end": start + len(substr), "text": substr})
    located.sort(key=lambda s: s["start"])
    for a, b in zip(located, located[1:]):
        if b["start"] < a["end"]:
            raise ValueError("overlapping spans returned by model")
    return located


def spans_to_bio(tokens_with_offsets, spans):
    """Assign a BIO tag to each token given a list of {"label", "start", "end"} spans."""
    tags = ["O"] * len(tokens_with_offsets)
    for span in spans:
        first = True
        for i, (_tok, s, e) in enumerate(tokens_with_offsets):
            if s >= span["end"]:
                break
            if e <= span["start"]:
                continue
            tags[i] = f"{'B' if first else 'I'}-{span['label']}"
            first = False
    return tags

## Prompt and structured output schema

We use a strict JSON schema (`response_format={"type": "json_schema", ...}`) so the model returns
the generated sentence plus its conditional spans as exact verbatim substrings — no free-form
parsing needed. Two system prompts (`SYSTEM_PROMPTS["marked"]` / `["unmarked"]`) drive the two
prompt variants described above; the shared schema and topic/register sampling stay the same for
both.

In [4]:
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "persian_conditional_example",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "A single Persian sentence (or short passage) containing exactly one conditional construction.",
                },
                "spans": {
                    "type": "array",
                    "description": "The conditional's protasis (IF) and apodosis (THEN) clauses.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "label": {
                                "type": "string",
                                "enum": ["IF", "THEN"],
                                "description": "IF = the condition clause (protasis). THEN = the consequent/main clause (apodosis).",
                            },
                            "text": {
                                "type": "string",
                                "description": "The exact verbatim substring of `text` covering this clause, character-for-character.",
                            },
                        },
                        "required": ["label", "text"],
                        "additionalProperties": False,
                    },
                },
            },
            "required": ["text", "spans"],
            "additionalProperties": False,
        },
    },
}

SYSTEM_PROMPTS = {
    "marked": (
        "You generate natural Persian (Farsi) training data for a conditional-clause detection "
        "task. Each response must be a single sentence or short passage containing exactly one "
        "conditional construction that uses an explicit conditional marker word, such as "
        "اگر, هرگاه, در صورتی که, چنانچه, یا به شرطی که (or a similar marker), pairing a marked "
        "protasis clause with a consequent clause. Vary sentence register (formal/colloquial), "
        "conditional type (real, hypothetical, counterfactual, generic/habitual), and clause "
        "order (protasis-first or apodosis-first) across responses."
    ),
    "unmarked": (
        "You generate natural Persian (Farsi) training data for a conditional-clause detection "
        "task. Each response must be a single sentence or short passage that expresses exactly "
        "one conditional relation WITHOUT using any explicit conditional marker word — do not use "
        "اگر, هرگاه, در صورتی که, چنانچه, به شرطی که, وگرنه, or any other overt conditional "
        "conjunction. Instead express the condition/consequence relation implicitly, e.g. through "
        "imperative + result juxtaposition (\"بخواب، حالت بهتر می‌شه\"), paratactic clause "
        "chaining, or subjunctive/future verb mood implying a hypothetical. The relation must "
        "still be clearly a condition -> consequence, just without the marker word. Vary sentence "
        "register (formal/colloquial), conditional type, and clause order across responses."
    ),
}

TOPICS = [
    "اقتصاد و قیمت‌ها", "آب‌وهوا", "سفر", "تحصیل و دانشگاه", "خانواده و روابط",
    "تکنولوژی", "سلامت", "ورزش", "محیط زیست", "کار و شغل", "غذا و آشپزی",
    "ترافیک شهری", "دوستی", "سیاست", "موسیقی و هنر",
]
REGISTERS = ["محاوره‌ای و غیررسمی", "رسمی و نوشتاری"]


def build_messages(variant):
    system_prompt = SYSTEM_PROMPTS[variant]
    topic = random.choice(TOPICS)
    register = random.choice(REGISTERS)
    user_prompt = (
        f"یک جمله فارسی با یک رابطه‌ی شرطی درباره‌ی «{topic}» تولید کن. "
        f"لحن جمله باید {register} باشد."
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

## Generation with validation and retries

`generate_example` takes an explicit `variant` (`"marked"` or `"unmarked"`) and records it on the
returned example as `prompt_variant`, so the generation loop below can alternate deterministically
between the two prompts and every example is traceable to the prompt that produced it.

In [5]:
def generate_example(client, config, variant):
    """Call the model with the given prompt variant, validate the returned spans, and return a
    fully-annotated example. Retries from scratch (fresh sample, same variant) up to
    config['max_retries'] times on validation failure."""
    last_err = None
    for attempt in range(1, config["max_retries"] + 1):
        try:
            response = client.chat.completions.create(
                model=config["model_name"],
                messages=build_messages(variant),
                temperature=config["temperature"],
                response_format=RESPONSE_FORMAT,
            )
            data = json.loads(response.choices[0].message.content)
            text = data["text"].strip()
            if not text:
                raise ValueError("empty text returned")

            spans = locate_spans(text, data["spans"])
            if not spans:
                raise ValueError("no spans returned")

            tokens_with_offsets = tokenize_with_offsets(text)
            tags = spans_to_bio(tokens_with_offsets, spans)

            return {
                "text": text,
                "tokens": [tok for tok, _s, _e in tokens_with_offsets],
                "tags": tags,
                "spans": spans,
                "prompt_variant": variant,
            }
        except Exception as exc:  # noqa: BLE001 - broad on purpose, we retry
            last_err = exc
            print(f"  [retry {attempt}/{config['max_retries']}, variant={variant}] {type(exc).__name__}: {exc}")

    raise RuntimeError(
        f"failed to generate a valid '{variant}' example after {config['max_retries']} attempts"
    ) from last_err

In [6]:
examples = []
variant_cycle = itertools.cycle(CONFIG["variants"])
for i in tqdm(range(CONFIG["num_examples"])):
    variant = next(variant_cycle)
    try:
        example = generate_example(client, CONFIG, variant)
        example["id"] = i
        examples.append(example)
    except RuntimeError as exc:
        print(f"Skipping example {i} ({variant}): {exc}")

print(f"Generated {len(examples)}/{CONFIG['num_examples']} examples")

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [36:47<00:00,  2.21s/it]

Generated 1000/1000 examples


## Inspect a sample

In [7]:
def show_example(example):
    print(f"[{example['prompt_variant']}] {example['text']}")
    for span in example["spans"]:
        print(f"  {span['label']:>4}: {span['text']!r}")
    print()
    for tok, tag in zip(example["tokens"], example["tags"]):
        marker = "" if tag == "O" else f"  <- {tag}"
        print(f"  {tok}{marker}")


if examples:
    show_example(examples[0])

# also show one of the other variant, if present
other = next((ex for ex in examples if ex["prompt_variant"] != examples[0]["prompt_variant"]), None)
if other:
    print("\n---\n")
    show_example(other)

[marked] اگه پیازو خوب تفت بدی، خورشتت حسابی جا می‌افته.
    IF: 'اگه پیازو خوب تفت بدی'
  THEN: 'خورشتت حسابی جا می\u200cافته.'

  اگه  <- B-IF
  پیازو  <- I-IF
  خوب  <- I-IF
  تفت  <- I-IF
  بدی  <- I-IF
  ،
  خورشتت  <- B-THEN
  حسابی  <- I-THEN
  جا  <- I-THEN
  می‌افته  <- I-THEN
  .  <- I-THEN

---

[unmarked] قیمت‌ها کاهش یابند، قدرت خرید خانوارها افزایش خواهد یافت.
    IF: 'قیمت\u200cها کاهش یابند'
  THEN: 'قدرت خرید خانوارها افزایش خواهد یافت'

  قیمت‌ها  <- B-IF
  کاهش  <- I-IF
  یابند  <- I-IF
  ،
  قدرت  <- B-THEN
  خرید  <- I-THEN
  خانوارها  <- I-THEN
  افزایش  <- I-THEN
  خواهد  <- I-THEN
  یافت  <- I-THEN
  .


## Save to JSONL

One example per line, CoNLL-style: `id`, `tokens`, `tags` for direct use in sequence-labeling
training, plus `text`, character-offset `spans`, and `prompt_variant` for reference/debugging and
downstream filtering by prompt type.

In [8]:
with open(CONFIG["output_path"], "w", encoding="utf-8") as f:
    for example in examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"Wrote {len(examples)} examples to {CONFIG['output_path']}")

Wrote 1000 examples to persian_conditionals_bio.jsonl


## Quick sanity stats

In [9]:
from collections import Counter

variant_counts = Counter(ex["prompt_variant"] for ex in examples)
num_if = sum(any(t.endswith("IF") for t in ex["tags"]) for ex in examples)
num_then = sum(any(t.endswith("THEN") for t in ex["tags"]) for ex in examples)
avg_tokens = sum(len(ex["tokens"]) for ex in examples) / len(examples) if examples else 0

print(f"Examples by prompt_variant:  {dict(variant_counts)}")
print(f"Examples with an IF span:    {num_if}/{len(examples)}")
print(f"Examples with a THEN span:   {num_then}/{len(examples)}")
print(f"Average tokens per example:  {avg_tokens:.1f}")

Examples by prompt_variant:  {'marked': 500, 'unmarked': 500}
Examples with an IF span:    1000/1000
Examples with a THEN span:   1000/1000
Average tokens per example:  13.1


In [10]:
with open(CONFIG["output_path"], encoding="utf-8") as f:
    string_read = f.read()
    reconstructions = [json.loads(part.strip()) for part in string_read.split("\n") if part.strip()]

In [11]:
reconstructions[1]

{'text': 'قیمت\u200cها کاهش یابند، قدرت خرید خانوارها افزایش خواهد یافت.',
 'tokens': ['قیمت\u200cها',
  'کاهش',
  'یابند',
  '،',
  'قدرت',
  'خرید',
  'خانوارها',
  'افزایش',
  'خواهد',
  'یافت',
  '.'],
 'tags': ['B-IF',
  'I-IF',
  'I-IF',
  'O',
  'B-THEN',
  'I-THEN',
  'I-THEN',
  'I-THEN',
  'I-THEN',
  'I-THEN',
  'O'],
 'spans': [{'label': 'IF',
   'start': 0,
   'end': 18,
   'text': 'قیمت\u200cها کاهش یابند'},
  {'label': 'THEN',
   'start': 20,
   'end': 56,
   'text': 'قدرت خرید خانوارها افزایش خواهد یافت'}],
 'prompt_variant': 'unmarked',
 'id': 1}